# Music Annotation Project
## Notebook 72R: Post-Lock Randomised Three-Way Performance-Usability Review

**Runtime: CPU.** Run only after Notebook 72 has closed its prediction lock.

This reviewer compares three locked page versions at identical scale:

- the **observed annotated page**;
- the **frozen V7.8 baseline**;
- the **unified V8.8 corrected page**.

Their assignment to **A, B and C is independently randomised per page** and remains sealed until every page judgement is complete and the review lock is closed.

### Why three-way rather than A/B
A three-way review directly answers two different questions that an A/B review conflates:

1. Does either correction improve on the annotated input?
2. Does unified V8.8 improve on, match or regress against the frozen V7.8 baseline?

The observed page is a control, not a production candidate. V7.8 is the incumbent comparator. V8.8 is the challenger.

### Review firewall
During initial review this notebook does **not** show clean truth, masks, difference maps, decisions, probabilities, filenames identifying the variants, or the sealed assignment key. Diagnostics remain unavailable until the review lock closes.

### Required reviewer outcome
For each page, record:

- best overall version: A, B, C or TIE;
- ranked order, allowing ties;
- performance-usability outcome;
- annotation removal;
- visible notation damage;
- visible residue;
- performability;
- confidence and notes.

This notebook does not rerun V8.8 and cannot change any locked page output.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gradio opencv-python-headless pandas numpy pillow

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, hmac, io, json, os, platform, secrets, shutil, sys, zipfile
import cv2, gradio as gr, numpy as np, pandas as pd
from PIL import Image

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NOTEBOOK72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
SOURCE_RUN_OVERRIDE=None  # Set to the exact completed Notebook 72 run if desired.
OUTPUT_ROOT=ROOT/'notebook72r_post_lock_review'
REVIEWER_CODE='DB'        # Replace with stable reviewer code before first launch.
REVIEW_REVISION='72R-v1-three-way'
RANDOM_SEED=720088
PUBLIC_LINK=True
REQUIRE_ALL_151=True

REQUIRED_SOURCE=['prediction_lock.json','notebook72_verdict.json','inputs/runtime_roster.csv','tables/page_output_hashes.csv','tables/runtime_page_summary.csv','pages']
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
SOURCE=Path(SOURCE_RUN_OVERRIDE) if SOURCE_RUN_OVERRIDE else latest_complete(NOTEBOOK72_ROOT,REQUIRED_SOURCE)
if SOURCE is None: raise RuntimeError('No complete Notebook 72 run found.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook72r_three_way_review'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; REVIEW=OUT/'review'; SEALED=OUT/'sealed'; DECODED=OUT/'decoded'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,REVIEW,SEALED,DECODED,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Source Notebook 72:',SOURCE)
print('Review output:',OUT)

## 2. Integrity helpers and source-lock verification

In [ ]:
def require(ok,msg):
    if not bool(ok): raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp'); t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str)); t.replace(path)
def read_gray(path):
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(a is not None,f'Unreadable image: {path}'); return a

def semantic_hash_matches(a,expected): return array_hash(a)==str(expected)

lock=json.loads((SOURCE/'prediction_lock.json').read_text())
verdict=json.loads((SOURCE/'notebook72_verdict.json').read_text())
require(lock.get('state')=='CLOSED','Notebook 72 prediction lock is not CLOSED.')
require(verdict.get('first_failed_gate') is None,'Notebook 72 has a failed gate.')
require(verdict.get('verdict')=='PASS_NOTEBOOK72_UNIFIED_V88_RUNTIME_COMPLETE_REVIEW_PENDING','Unexpected Notebook 72 verdict.')
roster=pd.read_csv(SOURCE/'inputs'/'runtime_roster.csv')
hashes=pd.read_csv(SOURCE/'tables'/'page_output_hashes.csv').set_index('page_key')
summary=pd.read_csv(SOURCE/'tables'/'runtime_page_summary.csv')
require(len(roster)==len(hashes)==len(summary),'Notebook 72 roster/hash/summary counts differ.')
if REQUIRE_ALL_151: require(len(roster)==151,f'Expected 151 pages; found {len(roster)}.')
require(lock['payload']['runtime_roster_sha256']==sha256_file(SOURCE/'inputs'/'runtime_roster.csv'),'Roster changed after lock.')
require(lock['payload']['page_output_hashes_sha256']==sha256_file(SOURCE/'tables'/'page_output_hashes.csv'),'Output-hash table changed after lock.')
source_identity={'source_run':str(SOURCE),'prediction_lock_id':lock['lock_id'],'operator_bundle_id':lock['payload']['operator_bundle_id'],'roster_sha256':sha256_file(SOURCE/'inputs'/'runtime_roster.csv'),'page_output_hashes_sha256':sha256_file(SOURCE/'tables'/'page_output_hashes.csv'),'pages':len(roster),'review_revision':REVIEW_REVISION,'reviewer_code':REVIEWER_CODE}
atomic_json(INPUTS/'source_identity.json',source_identity)
print(json.dumps(source_identity,indent=2))

## 3. Verify observed, V7.8 baseline and V8.8 corrected images

In [ ]:
page_records=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'; d=SOURCE/'pages'/key
    observed=read_gray(d/'observed.png'); baseline=read_gray(d/'baseline_v78.png'); corrected=read_gray(d/'corrected.png')
    require(observed.shape==baseline.shape==corrected.shape,f'{key}: variant shape mismatch.')
    h=hashes.loc[key]
    require(semantic_hash_matches(observed,h.observed_array_sha256),f'{key}: observed semantic hash mismatch.')
    require(semantic_hash_matches(baseline,h.baseline_array_sha256),f'{key}: baseline semantic hash mismatch.')
    require(semantic_hash_matches(corrected,h.corrected_array_sha256),f'{key}: corrected semantic hash mismatch.')
    page_records.append({'page_key':key,'page':r.page,'population':r.population,'observed_path':str(d/'observed.png'),'baseline_path':str(d/'baseline_v78.png'),'corrected_path':str(d/'corrected.png'),'shape':str(observed.shape),'observed_array_sha256':array_hash(observed),'baseline_array_sha256':array_hash(baseline),'corrected_array_sha256':array_hash(corrected)})
pages=pd.DataFrame(page_records)
pages.to_csv(INPUTS/'verified_review_roster.csv',index=False)
print('Verified locked variants:',len(pages),'pages x 3 images')

## 4. Create deterministic concealed A/B/C assignments and sealed key

In [ ]:
# The secret is stored only in sealed/ and is not displayed by the reviewer UI.
secret=secrets.token_bytes(32)
rng=np.random.default_rng(RANDOM_SEED)
assignments=[]
labels=['A','B','C']; variants=['OBSERVED','BASELINE_V78','CORRECTED_V88']
for order,(_,r) in enumerate(pages.iterrows(),1):
    perm=list(rng.permutation(variants))
    row={'review_order':order,'page_key':r.page_key,'page':r.page,'population':r.population}
    row.update({label:variant for label,variant in zip(labels,perm)})
    row['assignment_commitment']=hmac.new(secret,json.dumps(row,sort_keys=True).encode(),hashlib.sha256).hexdigest()
    assignments.append(row)
key=pd.DataFrame(assignments)
key.to_csv(SEALED/'review_key.csv',index=False)
atomic_json(SEALED/'review_key_seal.json',{'created_utc':datetime.now(timezone.utc).isoformat(),'review_revision':REVIEW_REVISION,'prediction_lock_id':lock['lock_id'],'rows':len(key),'review_key_sha256':sha256_file(SEALED/'review_key.csv'),'secret_b64':base64.b64encode(secret).decode(),'status':'SEALED_UNTIL_REVIEW_LOCK'})
public=key[['review_order','page_key','assignment_commitment']].copy(); public.to_csv(REVIEW/'review_queue_public.csv',index=False)
require(all(set(row)==set(variants) for row in key[['A','B','C']].to_numpy()),'Invalid A/B/C permutations.')
print('Randomised review queue created:',len(public))

## 5. Persistent response store and review-state contract

In [ ]:
RESPONSES=REVIEW/'responses.csv'; HISTORY=REVIEW/'response_history.jsonl'; REVIEW_LOCK=OUT/'review_lock.json'
RESPONSE_COLUMNS=['review_order','page_key','reviewer_code','review_revision','started_utc','submitted_utc','best_overall','rank_order','performance_outcome','annotation_removal','notation_damage','visible_residue','performability','confidence','notes','assignment_commitment','response_sha256']
if not RESPONSES.exists(): pd.DataFrame(columns=RESPONSE_COLUMNS).to_csv(RESPONSES,index=False)

def response_frame():
    q=pd.read_csv(RESPONSES,dtype=str).fillna('')
    require(not q.page_key.duplicated().any(),'Duplicate current response rows.')
    return q

def response_hash(row): return object_hash({k:row[k] for k in RESPONSE_COLUMNS if k!='response_sha256'})
def completed_keys(): return set(response_frame().page_key)
def review_closed(): return REVIEW_LOCK.exists()

def save_response(row):
    require(not review_closed(),'Review lock is closed; responses are immutable.')
    q=response_frame(); row['response_sha256']=response_hash(row)
    old=q[q.page_key==row['page_key']]
    event={'utc':datetime.now(timezone.utc).isoformat(),'event':'create' if old.empty else 'replace','page_key':row['page_key'],'new':row,'old':old.iloc[0].to_dict() if not old.empty else None}
    with open(HISTORY,'a') as f: f.write(json.dumps(event,sort_keys=True)+'\n')
    q=q[q.page_key!=row['page_key']]; q=pd.concat([q,pd.DataFrame([row])],ignore_index=True).sort_values('review_order'); q.to_csv(RESPONSES,index=False)
    return row['response_sha256']

## 6. Launch the blinded three-way reviewer

In [ ]:
LOOKUP=pages.set_index('page_key').to_dict('index'); KEY_PUBLIC=public.set_index('page_key').to_dict('index')
def variant_path(page_key,variant):
    r=LOOKUP[page_key]
    return Path({'OBSERVED':r['observed_path'],'BASELINE_V78':r['baseline_path'],'CORRECTED_V88':r['corrected_path']}[variant])
def render_variant(page_key,label):
    variant=key.set_index('page_key').loc[page_key,label]
    return read_gray(variant_path(page_key,variant))
def queue_choices():
    done=completed_keys(); return [x for x in public.sort_values('review_order').page_key if x not in done]
def page_status(): return f'Completed {len(completed_keys())} / {len(public)} | Remaining {len(public)-len(completed_keys())}'
def load_page(page_key):
    if not page_key: return None,None,None,'Select a page.',*[gr.update() for _ in range(10)]
    done=page_key in completed_keys(); msg=f'{page_status()} | Review item {int(KEY_PUBLIC[page_key]["review_order"])} | '+('Previously submitted; resubmission replaces current response and preserves history.' if done else 'Unsubmitted')
    return render_variant(page_key,'A'),render_variant(page_key,'B'),render_variant(page_key,'C'),msg,'TIE','A=B=C','REVIEW_REQUIRED','PARTIAL','UNCERTAIN','VISIBLE_RESIDUE','YES_WITH_MINOR_CONCERN','MEDIUM',''

def submit(page_key,best,rank,outcome,removal,damage,residue,performability,confidence,notes):
    require(page_key in LOOKUP,'No page selected.')
    required=[best,rank,outcome,removal,damage,residue,performability,confidence]
    if any(not str(x).strip() for x in required): raise gr.Error('Complete all required judgement fields.')
    now=datetime.now(timezone.utc).isoformat(); existing=response_frame(); prior=existing[existing.page_key==page_key]
    row={'review_order':int(KEY_PUBLIC[page_key]['review_order']),'page_key':page_key,'reviewer_code':REVIEWER_CODE,'review_revision':REVIEW_REVISION,'started_utc':prior.iloc[0].started_utc if not prior.empty else now,'submitted_utc':now,'best_overall':best,'rank_order':rank,'performance_outcome':outcome,'annotation_removal':removal,'notation_damage':damage,'visible_residue':residue,'performability':performability,'confidence':confidence,'notes':notes,'assignment_commitment':KEY_PUBLIC[page_key]['assignment_commitment']}
    digest=save_response(row); choices=queue_choices(); next_page=choices[0] if choices else None
    return gr.update(choices=choices,value=next_page),f'Saved {page_key} | response {digest[:12]}… | {page_status()}'

with gr.Blocks(title='Notebook 72R three-way blinded review') as app:
    gr.Markdown('# Notebook 72R — randomised A/B/C whole-page review\nJudge only the rendered pages. A/B/C identities remain concealed. Do not open the source run, sealed directory or diagnostics while reviewing.')
    with gr.Row():
        page=gr.Dropdown(choices=queue_choices(),value=(queue_choices()[0] if queue_choices() else None),label='Review item')
        status=gr.Textbox(value=page_status(),label='Status',interactive=False)
    with gr.Row():
        img_a=gr.Image(label='A',height=720); img_b=gr.Image(label='B',height=720); img_c=gr.Image(label='C',height=720)
    best=gr.Radio(['A','B','C','TIE'],value='TIE',label='Best overall version')
    rank=gr.Dropdown(['A>B>C','A>C>B','B>A>C','B>C>A','C>A>B','C>B>A','A=B>C','A=C>B','B=C>A','A>B=C','B>A=C','C>A=B','A=B=C'],value='A=B=C',label='Rank order, ties allowed')
    with gr.Row():
        outcome=gr.Radio(['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE','REVIEW_REQUIRED','UNUSABLE_ANNOTATION_REMAINS','UNUSABLE_PRINT_DAMAGE'],value='REVIEW_REQUIRED',label='Overall performance-usability outcome')
        performability=gr.Radio(['YES','YES_WITH_MINOR_CONCERN','NO'],value='YES_WITH_MINOR_CONCERN',label='Clean and trustworthy enough to perform from?')
    with gr.Row():
        removal=gr.Radio(['COMPLETE','PARTIAL','NONE','NOT_APPLICABLE'],value='PARTIAL',label='Annotation removal in best version')
        damage=gr.Radio(['NONE','UNCERTAIN','VISIBLE_MATERIAL_DAMAGE'],value='UNCERTAIN',label='Notation damage in best version')
        residue=gr.Radio(['NO_VISIBLE_RESIDUE','VISIBLE_RESIDUE','OBJECTIONABLE_RESIDUE','UNCERTAIN'],value='VISIBLE_RESIDUE',label='Residue in best version')
        confidence=gr.Radio(['HIGH','MEDIUM','LOW'],value='MEDIUM',label='Reviewer confidence')
    notes=gr.Textbox(label='Region/reason and notes',lines=3)
    save=gr.Button('Submit and load next',variant='primary')
    page.change(load_page,page,[img_a,img_b,img_c,status,best,rank,outcome,removal,damage,residue,performability,confidence,notes])
    app.load(load_page,page,[img_a,img_b,img_c,status,best,rank,outcome,removal,damage,residue,performability,confidence,notes])
    save.click(submit,[page,best,rank,outcome,removal,damage,residue,performability,confidence,notes],[page,status]).then(load_page,page,[img_a,img_b,img_c,status,best,rank,outcome,removal,damage,residue,performability,confidence,notes])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 7. Close the human-review lock

Run this cell **only when the UI reports 151 / 151 completed**. It validates every response, verifies each assignment commitment and freezes the response table. After closure, decoding is permitted and further edits are blocked.

In [ ]:
responses=response_frame()
require(len(responses)==len(public),f'Review incomplete: {len(responses)} / {len(public)}.')
require(set(responses.page_key)==set(public.page_key),'Response roster differs from public review queue.')
require((responses.reviewer_code==REVIEWER_CODE).all(),'Reviewer-code mismatch.')
require((responses.review_revision==REVIEW_REVISION).all(),'Review-revision mismatch.')
for _,r in responses.iterrows():
    require(r.response_sha256==response_hash(r.to_dict()),f'{r.page_key}: response hash mismatch.')
    require(r.assignment_commitment==KEY_PUBLIC[r.page_key]['assignment_commitment'],f'{r.page_key}: assignment commitment mismatch.')
lock_payload={'source_prediction_lock_id':lock['lock_id'],'source_operator_bundle_id':lock['payload']['operator_bundle_id'],'review_revision':REVIEW_REVISION,'reviewer_code':REVIEWER_CODE,'review_queue_sha256':sha256_file(REVIEW/'review_queue_public.csv'),'responses_sha256':sha256_file(RESPONSES),'response_history_sha256':sha256_file(HISTORY),'review_key_sha256':sha256_file(SEALED/'review_key.csv'),'responses':len(responses)}
review_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'state':'CLOSED','review_lock_id':object_hash(lock_payload),'payload':lock_payload}
atomic_json(REVIEW_LOCK,review_lock)
print('REVIEW LOCK CLOSED:',review_lock['review_lock_id'])

## 8. Decode A/B/C only after the review lock is closed

In [ ]:
require(review_closed(),'Review lock must be closed before decoding.')
review_lock=json.loads(REVIEW_LOCK.read_text()); require(review_lock['state']=='CLOSED','Review lock invalid.')
seal=json.loads((SEALED/'review_key_seal.json').read_text()); require(seal['review_key_sha256']==sha256_file(SEALED/'review_key.csv'),'Sealed review key changed.')
secret=base64.b64decode(seal['secret_b64']); key=pd.read_csv(SEALED/'review_key.csv')
for _,r in key.iterrows():
    row={k:r[k] for k in ['review_order','page_key','page','population','A','B','C']}
    expected=hmac.new(secret,json.dumps(row,sort_keys=True).encode(),hashlib.sha256).hexdigest()
    require(expected==r.assignment_commitment,f'{r.page_key}: concealed assignment commitment failed.')
responses=response_frame(); decoded=responses.merge(key,on=['review_order','page_key','assignment_commitment'],validate='one_to_one')

def chosen_variant(row):
    return 'TIE' if row.best_overall=='TIE' else row[row.best_overall]
def variant_position(rank,label):
    groups=rank.split('>')
    for i,g in enumerate(groups,1):
        if label in g.split('='): return i
    return None

def relation(row,left,right):
    label_left=next(k for k in ['A','B','C'] if row[k]==left); label_right=next(k for k in ['A','B','C'] if row[k]==right)
    pl=variant_position(row.rank_order,label_left); pr=variant_position(row.rank_order,label_right)
    return 'BETTER' if pl<pr else ('WORSE' if pl>pr else 'SAME')

decoded['best_variant']=decoded.apply(chosen_variant,axis=1)
decoded['v88_vs_baseline']=decoded.apply(lambda r:relation(r,'CORRECTED_V88','BASELINE_V78'),axis=1)
decoded['v88_vs_observed']=decoded.apply(lambda r:relation(r,'CORRECTED_V88','OBSERVED'),axis=1)
decoded['baseline_vs_observed']=decoded.apply(lambda r:relation(r,'BASELINE_V78','OBSERVED'),axis=1)
decoded.to_csv(DECODED/'performance_usability_reviews.csv',index=False)
print(decoded[['page_key','best_variant','v88_vs_baseline','performance_outcome','notation_damage']].head())

## 9. Produce the final Notebook 72 decision tables

In [ ]:
def counts_table(frame,column):
    q=frame.groupby(column,dropna=False).size().reset_index(name='pages'); q['fraction']=q.pages/len(frame); return q
summary_tables=[]
for c in ['best_variant','v88_vs_baseline','v88_vs_observed','baseline_vs_observed','performance_outcome','annotation_removal','notation_damage','visible_residue','performability','confidence']:
    q=counts_table(decoded,c); q.insert(0,'measure',c); q=q.rename(columns={c:'level'}); summary_tables.append(q)
review_summary=pd.concat(summary_tables,ignore_index=True); review_summary.to_csv(DECODED/'performance_usability_summary.csv',index=False)

source_summary=(decoded.groupby('population',as_index=False).agg(pages=('page_key','size'),v88_better_than_baseline=('v88_vs_baseline',lambda x:int((x=='BETTER').sum())),v88_same_as_baseline=('v88_vs_baseline',lambda x:int((x=='SAME').sum())),v88_worse_than_baseline=('v88_vs_baseline',lambda x:int((x=='WORSE').sum())),performance_ready=('performance_outcome',lambda x:int(x.isin(['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE']).sum())),review_required=('performance_outcome',lambda x:int((x=='REVIEW_REQUIRED').sum())),unusable_annotation=('performance_outcome',lambda x:int((x=='UNUSABLE_ANNOTATION_REMAINS').sum())),unusable_print_damage=('performance_outcome',lambda x:int((x=='UNUSABLE_PRINT_DAMAGE').sum())),material_damage=('notation_damage',lambda x:int((x=='VISIBLE_MATERIAL_DAMAGE').sum())),objectionable_residue=('visible_residue',lambda x:int((x=='OBJECTIONABLE_RESIDUE').sum()))))
source_summary.to_csv(DECODED/'results_by_source_group.csv',index=False)

failure=decoded[(decoded.performance_outcome.isin(['REVIEW_REQUIRED','UNUSABLE_ANNOTATION_REMAINS','UNUSABLE_PRINT_DAMAGE']))|(decoded.notation_damage=='VISIBLE_MATERIAL_DAMAGE')|(decoded.visible_residue=='OBJECTIONABLE_RESIDUE')|(decoded.v88_vs_baseline=='WORSE')].copy()
failure['failure_reasons']=failure.apply(lambda r:'|'.join([x for x,ok in [('REVIEW_REQUIRED',r.performance_outcome=='REVIEW_REQUIRED'),('ANNOTATION_REMAINS',r.performance_outcome=='UNUSABLE_ANNOTATION_REMAINS'),('PRINT_DAMAGE',r.performance_outcome=='UNUSABLE_PRINT_DAMAGE'),('VISIBLE_MATERIAL_DAMAGE',r.notation_damage=='VISIBLE_MATERIAL_DAMAGE'),('OBJECTIONABLE_RESIDUE',r.visible_residue=='OBJECTIONABLE_RESIDUE'),('V88_WORSE_THAN_BASELINE',r.v88_vs_baseline=='WORSE')] if ok]),axis=1)
failure.to_csv(DECODED/'failure_and_exception_register.csv',index=False)

# G7-G9 are evidence statements, not invented numerical promotion thresholds.
gates={'G7_structural_safety_evidence':{'material_damage_pages':int((decoded.notation_damage=='VISIBLE_MATERIAL_DAMAGE').sum()),'unusable_print_damage_pages':int((decoded.performance_outcome=='UNUSABLE_PRINT_DAMAGE').sum()),'status':'EVIDENCE_REPORTED_THRESHOLD_NOT_PREDECLARED'},'G8_cosmetic_usefulness_evidence':{'performance_ready_pages':int(decoded.performance_outcome.isin(['PERFORMANCE_READY','PERFORMANCE_READY_WITH_MINOR_RESIDUE']).sum()),'unusable_annotation_pages':int((decoded.performance_outcome=='UNUSABLE_ANNOTATION_REMAINS').sum()),'objectionable_residue_pages':int((decoded.visible_residue=='OBJECTIONABLE_RESIDUE').sum()),'status':'EVIDENCE_REPORTED_THRESHOLD_NOT_PREDECLARED'},'G9_source_stability_evidence':{'source_groups':len(source_summary),'groups_with_v88_worse':int((source_summary.v88_worse_than_baseline>0).sum()),'status':'EVIDENCE_REPORTED_THRESHOLD_NOT_PREDECLARED'},'G10_auditability':{'source_prediction_lock_id':lock['lock_id'],'review_lock_id':review_lock['review_lock_id'],'decoded_rows':len(decoded),'status':'PASS'}}

integrity_pass=(len(decoded)==151 and gates['G10_auditability']['status']=='PASS')
if not integrity_pass: final_verdict='FAIL_NOTEBOOK72R_REVIEW_INVALID'
elif gates['G7_structural_safety_evidence']['material_damage_pages']>0 or gates['G7_structural_safety_evidence']['unusable_print_damage_pages']>0: final_verdict='NOTEBOOK72R_COMPLETE_STRUCTURAL_FAILURES_REQUIRE_INVESTIGATION'
elif gates['G8_cosmetic_usefulness_evidence']['unusable_annotation_pages']>0 or gates['G8_cosmetic_usefulness_evidence']['objectionable_residue_pages']>0: final_verdict='NOTEBOOK72R_COMPLETE_TARGETED_USE_OR_REVIEW_REQUIRED'
else: final_verdict='NOTEBOOK72R_COMPLETE_EVIDENCE_SUPPORTS_SUPERVISED_PILOT_CONSIDERATION'

record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':final_verdict,'promotion_authority':False,'source_prediction_lock_id':lock['lock_id'],'review_lock_id':review_lock['review_lock_id'],'operator_bundle_id':lock['payload']['operator_bundle_id'],'pages':len(decoded),'reviewer_code':REVIEWER_CODE,'review_revision':REVIEW_REVISION,'v88_vs_baseline':decoded.v88_vs_baseline.value_counts().to_dict(),'best_variant':decoded.best_variant.value_counts().to_dict(),'performance_outcome':decoded.performance_outcome.value_counts().to_dict(),'gates':gates,'interpretation':'The review is complete and decoded. Numerical promotion thresholds for G7-G9 were not predeclared, so this notebook reports evidence and a conservative disposition rather than inventing a retrospective production pass.'}
atomic_json(DECODED/'final_notebook72_verdict.json',record)
lines=['NOTEBOOK 72R - POST-LOCK RANDOMISED THREE-WAY REVIEW','',f'Verdict: {final_verdict}',f'Source prediction lock: {lock["lock_id"]}',f'Review lock: {review_lock["review_lock_id"]}',f'Pages reviewed: {len(decoded)}',f'Reviewer: {REVIEWER_CODE}','',f'V8.8 better / same / worse than V7.8: {int((decoded.v88_vs_baseline=="BETTER").sum())} / {int((decoded.v88_vs_baseline=="SAME").sum())} / {int((decoded.v88_vs_baseline=="WORSE").sum())}',f'Performance-ready or ready-with-minor-residue: {int(decoded.performance_outcome.isin(["PERFORMANCE_READY","PERFORMANCE_READY_WITH_MINOR_RESIDUE"]).sum())}',f'Visible material-damage pages: {int((decoded.notation_damage=="VISIBLE_MATERIAL_DAMAGE").sum())}',f'Unusable annotation-remains pages: {int((decoded.performance_outcome=="UNUSABLE_ANNOTATION_REMAINS").sum())}',f'Unusable print-damage pages: {int((decoded.performance_outcome=="UNUSABLE_PRINT_DAMAGE").sum())}',f'Failure/exception register rows: {len(failure)}','','No automatic production promotion. G7-G9 numerical thresholds were not retrospectively invented.']
(DECODED/'final_executive_summary.txt').write_text('\n'.join(lines)); print('\n'.join(lines))

## 10. Curated final upload package

In [ ]:
required=[OUT/'review_lock.json',INPUTS/'source_identity.json',REVIEW/'review_queue_public.csv',REVIEW/'responses.csv',REVIEW/'response_history.jsonl',SEALED/'review_key.csv',SEALED/'review_key_seal.json',DECODED/'performance_usability_reviews.csv',DECODED/'performance_usability_summary.csv',DECODED/'results_by_source_group.csv',DECODED/'failure_and_exception_register.csv',DECODED/'final_notebook72_verdict.json',DECODED/'final_executive_summary.txt']
for p in required: require(p.exists(),f'Missing final artefact: {p}')
manifest=[]
for src in required:
    dst=UPLOAD/src.name; shutil.copy2(src,dst); manifest.append({'file':dst.name,'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 72R post-lock randomised three-way review package. Start with final_executive_summary.txt and final_notebook72_verdict.json. review_key.csv is included only because the review lock is closed. No production promotion is granted.\n')
with zipfile.ZipFile(UPLOAD/'notebook72r_final_review_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip': z.write(p,p.name)
print('Final review bundle:',UPLOAD/'notebook72r_final_review_bundle.zip')

## Decision boundary

Notebook 72R completes the missing human evidence step. It compares the annotated control, frozen V7.8 incumbent and unified V8.8 challenger without exposing their identities during judgement.

The notebook deliberately does not invent retrospective numerical release gates. Its final disposition is evidence-based and conservative:

- visible material print damage requires investigation;
- objectionable residue or unusable remaining annotation supports targeted use or continued review;
- a clean review may support consideration of a controlled supervised pilot;
- no outcome automatically authorises production deployment.